In [37]:
import os
import polars as pl
from towbintools.foundation.image_handling import read_tiff_file
import numpy as np
import yaml
import matplotlib.pyplot as plt
from tifffile import imwrite
from sklearn.model_selection import train_test_split
import shutil

SEED = 42

project_path = "/mnt/towbin.data/shared/spsalmon/towbinlab_cell_type_database/20251023_115945_091_ZIVA_60x_397_405_yap_dynamics/20260612_project"
annotation_path = os.path.join(project_path, "annotations", "annotations.csv")
annotations_df = pl.read_csv(annotation_path)

CHANNELS = [0, 1]
project_yaml_path = os.path.join(project_path, "project.yaml")
project = yaml.safe_load(open(project_yaml_path, "r"))

classes = project["classes"]
add_other = True
if add_other:
    classes.append("other")
class_to_index = {class_name: i + 1 for i, class_name in enumerate(classes)}
print(f"Classes: {classes}, Class to index: {class_to_index}")

panoptic_masks_output = os.path.join(project_path, "panoptic_masks")
data_sets_path = os.path.join(project_path, "data_sets")

os.makedirs(panoptic_masks_output, exist_ok=True)
os.makedirs(data_sets_path, exist_ok=True)

Classes: ['epidermis', 'intestine', 'other'], Class to index: {'epidermis': 1, 'intestine': 2, 'other': 3}


In [38]:
for row in annotations_df.iter_rows(named=True):
    segmentation_path, annotation_path = row['Segmentation'], row['Annotation']
    segmentation_path = segmentation_path.replace("//izbkingston", "/mnt").replace("\\", "/")
    annotation_path = annotation_path.replace("//izbkingston", "/mnt").replace("\\", "/")
    annotation = pl.read_csv(annotation_path)
    mask = read_tiff_file(segmentation_path)

    panoptic_mask = np.zeros_like(mask, dtype=np.uint8)

    for i, plane in enumerate(mask):
        plane_annotation = annotation.filter(pl.col("Z") == i)

        if np.sum(plane) == 0:
            continue
        for label in np.unique(plane):
            if label == 0:
                continue
            label_annotation = plane_annotation.filter(pl.col("Label") == label)
            if label_annotation.is_empty():
                if add_other:
                    panoptic_mask[i][mask[i] == label] = class_to_index["other"]
            else:
                class_name = label_annotation[0, "Class"]
                panoptic_mask[i][mask[i] == label] = class_to_index[class_name]

    output_path = os.path.join(panoptic_masks_output, os.path.basename(segmentation_path))
    imwrite(output_path, panoptic_mask, compression="zlib")

In [39]:
train_set_path = os.path.join(data_sets_path, "train")
val_set_path = os.path.join(data_sets_path, "val")
os.makedirs(train_set_path, exist_ok=True)
os.makedirs(val_set_path, exist_ok=True)

train_annotations, val_annotations = train_test_split(annotations_df, test_size=0.2, random_state=SEED)

print(f"Train set size: {len(train_annotations)}, Validation set size: {len(val_annotations)}")

train_dataset = []
for row in train_annotations.iter_rows(named=True):
    raw_path = row['Reference'].replace("//izbkingston", "/mnt").replace("\\", "/")
    panoptic_mask_path = os.path.join(panoptic_masks_output, os.path.basename(raw_path))
    raw_dest_path = os.path.join(train_set_path, "image", os.path.basename(raw_path))
    panoptic_dest_path = os.path.join(train_set_path, "mask", os.path.basename(panoptic_mask_path))
    os.makedirs(os.path.dirname(raw_dest_path), exist_ok=True)
    os.makedirs(os.path.dirname(panoptic_dest_path), exist_ok=True)

    raw, panoptic_mask = read_tiff_file(raw_path, channels_to_keep=CHANNELS), read_tiff_file(panoptic_mask_path)
    for i, raw_plane, mask_plane in zip(range(raw.shape[0]), raw, panoptic_mask):
        raw_plane_dest_path = os.path.join(train_set_path, "image", f"{os.path.splitext(os.path.basename(raw_path))[0]}_z{i}.tiff")
        mask_plane_dest_path = os.path.join(train_set_path, "mask", f"{os.path.splitext(os.path.basename(panoptic_mask_path))[0]}_z{i}.tiff")
        imwrite(raw_plane_dest_path, raw_plane, compression="zlib")
        imwrite(mask_plane_dest_path, mask_plane, compression="zlib")

        train_dataset.append({'image': raw_plane_dest_path, 'mask': mask_plane_dest_path})

train_dataset_df = pl.DataFrame(train_dataset)

val_dataset = []

for row in val_annotations.iter_rows(named=True):
    raw_path = row['Reference'].replace("//izbkingston", "/mnt").replace("\\", "/")
    panoptic_mask_path = os.path.join(panoptic_masks_output, os.path.basename(raw_path))
    raw_dest_path = os.path.join(val_set_path, "image", os.path.basename(raw_path))
    panoptic_dest_path = os.path.join(val_set_path, "mask", os.path.basename(panoptic_mask_path))
    os.makedirs(os.path.dirname(raw_dest_path), exist_ok=True)
    os.makedirs(os.path.dirname(panoptic_dest_path), exist_ok=True)
    
    raw, panoptic_mask = read_tiff_file(raw_path, channels_to_keep = CHANNELS), read_tiff_file(panoptic_mask_path)
    for i, raw_plane, mask_plane in zip(range(raw.shape[0]), raw, panoptic_mask):
        raw_plane_dest_path = os.path.join(val_set_path, "image", f"{os.path.splitext(os.path.basename(raw_path))[0]}_z{i}.tiff")
        mask_plane_dest_path = os.path.join(val_set_path, "mask", f"{os.path.splitext(os.path.basename(panoptic_mask_path))[0]}_z{i}.tiff")
        imwrite(raw_plane_dest_path, raw_plane, compression="zlib")
        imwrite(mask_plane_dest_path, mask_plane, compression="zlib")
        val_dataset.append({'image': raw_plane_dest_path, 'mask': mask_plane_dest_path})

val_dataset_df = pl.DataFrame(val_dataset)

train_dataset_df.write_csv(os.path.join(data_sets_path, "train_dataset.csv"))
val_dataset_df.write_csv(os.path.join(data_sets_path, "val_dataset.csv"))

Train set size: 24, Validation set size: 6
